In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
data_path = Path("../../Data/Original_Data")
mort = 'Complications_and_Deaths-Hospital.csv'
readm = 'Unplanned_Hospital_Visits-Hospital.csv'
safety = 'Healthcare_Associated_Infections-Hospital.csv'
cost = 'Medicare_Hospital_Spending_by_Claim.csv'
hospital = 'Hospital_General_Information.csv'

In [3]:
# Reading in the Complications and Deaths file. Two of the measures, COMP_HIP_KNEE and PSI_90, will be combined with the infections
# data to obtain a Safety score. The other measures will be used for the Mortality score.


df_mort_long = pd.read_csv(data_path / mort, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_mort_long.columns = (df_mort_long.columns.str.replace(' ', '_'))

df_mort_long = df_mort_long[df_mort_long['Measure_ID'].isin(['COMP_HIP_KNEE',
                                                'MORT_30_AMI',
                                                'MORT_30_CABG',
                                                'MORT_30_COPD',
                                                'MORT_30_HF',
                                                'MORT_30_PN',
                                                'MORT_30_STK',
                                                'PSI_04',
                                                'PSI_90'])]
df_mort_long['Score'] = df_mort_long['Score'].replace('Not Available', np.nan)
df_mort_long = df_mort_long.astype({'Score': 'float'})

In [4]:
df_mort = df_mort_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_mort = df_mort.reset_index()
df_mort = df_mort.set_index('Facility_ID')
df_mort.shape

(4789, 9)

In [5]:
# Reading in the Unplanned Hospital Visits file. These measures will be used for the Readmissions score.

df_readm_long = pd.read_csv(data_path / readm, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_readm_long.columns = (df_readm_long.columns.str.replace(' ', '_'))
df_readm_long = df_readm_long[df_readm_long['Measure_ID'].isin (['EDAC_30_AMI',
                                                                 'EDAC_30_HF',
                                                                 'EDAC_30_PN',
                                                                 'READM_30_CABG',
                                                                 'READM_30_COPD',
                                                                 'READM_30_HIP_KNEE',
                                                                 'Hybrid_HWR',
                                                                 'OP_32',
                                                                 'OP_35_ADM',
                                                                 'OP_35_ED',
                                                                 'OP_36'])]
df_readm_long['Score'] = df_readm_long['Score'].replace('Not Available', np.nan)
df_readm_long = df_readm_long.astype({'Score': 'float'})

In [6]:
df_readm_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 52679 entries, 0 to 67044
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Facility_ID   52679 non-null  object 
 1   Measure_ID    52679 non-null  object 
 2   Measure_Name  52679 non-null  object 
 3   Score         26662 non-null  float64
dtypes: float64(1), object(3)
memory usage: 2.0+ MB


In [7]:
df_readm = df_readm_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_readm = df_readm.reset_index()
df_readm = df_readm.set_index('Facility_ID')

In [8]:
df_readm.shape

(4789, 11)

In [9]:
# Reading in the Infections file. These measures (along with COMP_HIP_KNEE and PSI_90 from the Complications and Deaths file)
# will be used for the Safety score.

df_safety_long = pd.read_csv(data_path / safety, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_safety_long.columns = (df_safety_long.columns.str.replace(' ', '_'))
df_safety_long = df_safety_long[df_safety_long['Measure_ID'].isin(['HAI_1_SIR',
                                                  'HAI_2_SIR',
                                                  'HAI_3_SIR',
                                                  'HAI_4_SIR',
                                                  'HAI_5_SIR',
                                                  'HAI_6_SIR'])]
df_safety_long['Score'] = df_safety_long['Score'].replace('Not Available',np.nan)
df_safety_long = df_safety_long.astype({'Score': 'float'})

In [10]:
df_safety_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28734 entries, 5 to 172403
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Facility_ID   28734 non-null  object 
 1   Measure_ID    28734 non-null  object 
 2   Measure_Name  28734 non-null  object 
 3   Score         11272 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.1+ MB


In [11]:
df_safety = df_safety_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_safety = df_safety.reset_index()
df_safety = df_safety.set_index('Facility_ID')
#df_safety.head(5)

In [12]:
# Moving COMP_HIP_KNEE and PSI_90 from the mort df to the safety df
safety_cols = ['COMP_HIP_KNEE', 'PSI_90']
df_safety[safety_cols] = df_mort[safety_cols]
df_mort = df_mort.drop(columns = safety_cols) 

In [13]:
df_safety.shape

(4789, 8)

### Now that the data is separated into the 3 Category Groups, z-scores will be calculated for each measure individually and then a composite category z-score will be calculated for every hospital.  

#### Z-scores are used so that measures with vastly different ranges can be compared with one another
Z-score = (Hospital score - population mean) / population standard deviation.  Because all measures in these groups have smaller values as better, -1 will be multiplied so that all z-scores show higher values as better.

In [14]:
# Z-scores for all individual measures
mort_measures = ['MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04']

readm_measures = ['EDAC_30_AMI','EDAC_30_HF',
                     'EDAC_30_PN','READM_30_CABG',
                     'READM_30_COPD','READM_30_HIP_KNEE',
                     'Hybrid_HWR','OP_32','OP_35_ADM',
                     'OP_35_ED','OP_36']

safety_measures = ['HAI_1_SIR','HAI_2_SIR',
                      'HAI_3_SIR','HAI_4_SIR',
                      'HAI_5_SIR','HAI_6_SIR', 
                      'COMP_HIP_KNEE', 'PSI_90']

mort_z = (-1* (df_mort[mort_measures] - df_mort[mort_measures].mean())) / df_mort[mort_measures].std()
mort_z.columns = [m + '_Z' for m in mort_measures]
df_mort = pd.concat([df_mort, mort_z], axis = 1)

readm_z = (-1*(df_readm[readm_measures] - df_readm[readm_measures].mean())) / df_readm[readm_measures].std()
readm_z.columns = [r + '_Z' for r in readm_measures]
df_readm = pd.concat([df_readm, readm_z], axis = 1)

safety_z = (-1*(df_safety[safety_measures] - df_safety[safety_measures].mean())) / df_safety[safety_measures].std()
safety_z.columns = [s + '_Z' for s in safety_measures]
df_safety = pd.concat([df_safety, safety_z], axis = 1)

In [15]:
#df_mort.head(5)
#df_readm.head(5)
#df_safety.head(5)

### For each hospital, an average z-score is calculated for each category group

In [16]:

z_mort_measures = ['MORT_30_AMI_Z','MORT_30_CABG_Z', 
                    'MORT_30_COPD_Z','MORT_30_HF_Z',
                    'MORT_30_PN_Z','MORT_30_STK_Z','PSI_04_Z']

z_readm_measures = ['EDAC_30_AMI_Z','EDAC_30_HF_Z',
                     'EDAC_30_PN_Z','READM_30_CABG_Z',
                     'READM_30_COPD_Z','READM_30_HIP_KNEE_Z',
                     'Hybrid_HWR_Z','OP_32_Z','OP_35_ADM_Z',
                     'OP_35_ED_Z','OP_36_Z']

z_safety_measures = ['HAI_1_SIR_Z','HAI_2_SIR_Z',
                      'HAI_3_SIR_Z','HAI_4_SIR_Z',
                      'HAI_5_SIR_Z','HAI_6_SIR_Z', 
                      'COMP_HIP_KNEE_Z', 'PSI_90_Z']

df_mort['z_mean_mort'] = df_mort[z_mort_measures].mean(axis=1)
df_readm['z_mean_readm'] = df_readm[z_readm_measures].mean(axis=1)
df_safety['z_mean_safety'] = df_safety[z_safety_measures].mean(axis=1)


In [19]:
df_safety.head(5)

,HAI_1_SIR,HAI_2_SIR,HAI_3_SIR,HAI_4_SIR,HAI_5_SIR,HAI_6_SIR,COMP_HIP_KNEE,PSI_90,HAI_1_SIR_Z,HAI_2_SIR_Z,HAI_3_SIR_Z,HAI_4_SIR_Z,HAI_5_SIR_Z,HAI_6_SIR_Z,COMP_HIP_KNEE_Z,PSI_90_Z,z_mean_safety
Facility_ID,,,,,,,,,,,,,,,,,
010001,0.418,0.125,0.292,NaN,0.390,0.395,3.2,0.95,0.287385,0.748472,0.802655,NaN,0.512313,0.001343,0.570855,0.242668,0.452242
010005,1.430,1.054,0.804,NaN,2.431,0.430,3.0,0.97,-1.217619,-1.013994,0.071405,NaN,-2.881029,-0.078448,0.845263,0.139862,-0.590651
010006,0.000,0.105,0.000,0.0,0.200,0.102,4.7,1.14,0.909018,0.786415,1.219696,1.16011,0.828205,0.669310,-1.487205,-0.733989,0.418945
010007,NaN,NaN,NaN,NaN,NaN,0.772,NaN,1.06,NaN,NaN,NaN,NaN,NaN,-0.858124,NaN,-0.322765,-0.590444
010008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Now a master z-score is calculated from all of the hospitals' averages

In [21]:
df_mort['master_mort_z'] = (df_mort['z_mean_mort'] - df_mort['z_mean_mort'].mean()) / df_mort['z_mean_mort'].std()
df_readm['master_readm_z'] = (df_readm['z_mean_readm'] - df_readm['z_mean_readm'].mean()) / df_readm['z_mean_readm'].std()
df_safety['master_safety_z'] = (df_safety['z_mean_safety'] - df_safety['z_mean_safety'].mean()) / df_safety['z_mean_safety'].std()

In [29]:
df_hospital = pd.read_csv(data_path / hospital, usecols = ['Facility ID', 'Facility Name',
                                                           'City/Town','State','ZIP Code',
                                                           'Hospital Type','Hospital Ownership',
                                                           'Emergency Services','Count of Facility MORT Measures',
                                                           'Count of Facility Safety Measures',
                                                           'Count of Facility READM Measures',
                                                           'Count of Facility Pt Exp Measures',
                                                           'Count of Facility TE Measures'])
df_hospital.columns = (df_hospital.columns.str.replace(' ', '_').str.replace('/', '_'))

hospital_counters = ['Count_of_Facility_MORT_Measures','Count_of_Facility_Safety_Measures',
                     'Count_of_Facility_READM_Measures','Count_of_Facility_Pt_Exp_Measures',
                     'Count_of_Facility_TE_Measures']

df_hospital[hospital_counters] = df_hospital[hospital_counters].replace('Not Available', np.nan)
df_hospital = df_hospital.astype({'Count_of_Facility_MORT_Measures': 'float','Count_of_Facility_Safety_Measures': 'float',
                     'Count_of_Facility_READM_Measures': 'float','Count_of_Facility_Pt_Exp_Measures': 'float',
                     'Count_of_Facility_TE_Measures': 'float'})

df_hospital.head(5)

,Facility_ID,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Count_of_Facility_MORT_Measures,Count_of_Facility_Safety_Measures,Count_of_Facility_READM_Measures,Count_of_Facility_Pt_Exp_Measures,Count_of_Facility_TE_Measures
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,7.0,7.0,11.0,8.0,11.0
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,6.0,7.0,9.0,8.0,12.0
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,Acute Care Hospitals,Proprietary,Yes,7.0,8.0,9.0,8.0,10.0
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,Acute Care Hospitals,Voluntary non-profit - Private,Yes,3.0,3.0,7.0,8.0,7.0
4,010008,CRENSHAW COMMUNITY HOSPITAL,LUVERNE,AL,36049,Acute Care Hospitals,Proprietary,Yes,1.0,NaN,2.0,NaN,6.0


In [30]:
df_hospital.describe()

,ZIP_Code,Count_of_Facility_MORT_Measures,Count_of_Facility_Safety_Measures,Count_of_Facility_READM_Measures,Count_of_Facility_Pt_Exp_Measures,Count_of_Facility_TE_Measures
count,5426.000000,3640.000000,3353.000000,4266.000000,3151.0,4486.000000
mean,53785.590675,4.389011,4.749478,6.267932,8.0,7.964556
std,27067.251164,2.127120,2.465850,3.159561,0.0,2.983394
min,603.000000,1.000000,1.000000,1.000000,8.0,1.000000
25%,32782.500000,2.000000,2.000000,3.000000,8.0,6.000000
50%,55063.000000,5.000000,5.000000,6.000000,8.0,9.000000
75%,76104.000000,6.000000,7.000000,9.000000,8.0,10.000000
max,99929.000000,7.000000,8.000000,11.000000,8.0,12.000000


In [ ]:
# next step: define if columns meet thresholds for being included
# df['new_col'] = np.where(df['col'] >= 3, 'high', 'low')